# Heat Equation Inverse with DeepXDE
I attempt to solve an inverse problem of the Heat Equation with DeepXDE.

In [1]:
import deepxde as dde
import numpy as np
import torch

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


In [3]:
# define the domain
geom = dde.geometry.Interval(0.0, 1.0) # x from 0 to 1
timedomain = dde.geometry.TimeDomain(0.0, 1.0) # t from 0 to 1
geomtime = dde.geometry.GeometryXTime(geom, timedomain) # x vs t

# 1D Wave Equation
d2u/dt2 = c2 d2u/dx2

In [13]:
# Define the PDE
c = dde.Variable(0.5) # wave speed, to be trained

def wave_pde(x, u):
    u_tt = dde.grad.hessian(u, x, i=0, j=1) # with respect to t
    u_xx = dde.grad.hessian(u, x, i=0, j=0) # with respect to x
    return u_tt - c**2 * u_xx

def initial_condition(x):
    return np.sin(np.pi * x[:, 0:1])  # u(x,0) = sin(pi*x)

ic = dde.icbc.IC(
    geomtime,
    initial_condition,
    lambda _, on_initial: on_initial
)

# boundary conditions
bc_left = dde.icbc.DirichletBC(
    geomtime, lambda x: 0.0, lambda x, on_boundary: on_boundary and np.isclose(x[0], 0.0)
)
bc_right = dde.icbc.DirichletBC(
    geomtime, lambda x: 0.0, lambda x, on_boundary: on_boundary and np.isclose(x[0], 1.0)
)

In [15]:
# generate observational data (will need real solution class)

c_true = 0.1

def true_solution(x):
    return np.sin(np.pi * x[:, 0:1]) * np.cos(c_true * np.pi * x[:, 1:2])

N = 100

# bunch of x and t points
x_obs = np.random.rand(N, 1)
t_obs = np.random.rand(N, 1)
# combine:
X_obs = np.hstack((x_obs, t_obs))

# solution at those points
u_obs = true_solution(X_obs) # don't need to split because it's already combined

# add some noise
noise_level = 0.01
u_obs += noise_level * np.random.randn(*u_obs.shape)

# final observational data
observe_u = dde.icbc.PointSetBC(X_obs, u_obs)

In [25]:
# data object
data = dde.data.TimePDE(
    geomtime,
    wave_pde,
    [ic, bc_left, bc_right], # removed observe_u for now because it's cooked
    num_domain=10000,
    num_boundary=200,
    num_initial=200
)

# neural network
net = dde.nn.FNN(
    [2] + [50] * 3 + [1],
    activation="tanh",
    kernel_initializer="Glorot normal"
)

In [23]:
model = dde.Model(data, net)

model.compile(
    optimizer="adam",
    lr=1e-3,
    external_trainable_variables=[c]
)

Compiling model...
'compile' took 0.000152 s



In [24]:
model.train(epochs=5000)

print("Learned c:", c.item())

Training model...

Step      Train loss                                  Test loss                                   Test metric
0         [1.05e-03, 4.34e-01, 1.24e-02, 4.15e-02]    [1.05e-03, 4.34e-01, 1.24e-02, 4.15e-02]    []  
1000      [3.77e-05, 1.34e-04, 4.32e-05, 5.30e-05]    [3.77e-05, 1.34e-04, 4.32e-05, 5.30e-05]    []  
2000      [1.10e-05, 1.11e-05, 7.18e-06, 2.14e-05]    [1.10e-05, 1.11e-05, 7.18e-06, 2.14e-05]    []  
3000      [4.95e-06, 3.29e-05, 1.86e-06, 9.48e-05]    [4.95e-06, 3.29e-05, 1.86e-06, 9.48e-05]    []  
4000      [3.22e-06, 4.39e-06, 1.77e-06, 7.93e-06]    [3.22e-06, 4.39e-06, 1.77e-06, 7.93e-06]    []  
5000      [2.86e-06, 1.62e-05, 6.57e-07, 5.09e-05]    [2.86e-06, 1.62e-05, 6.57e-07, 5.09e-05]    []  

Best model at step 4000:
  train loss: 1.73e-05
  test loss: 1.73e-05
  test metric: []

'train' took 321.401018 s

Learned c: 0.012296395376324654
